# Partei-Autorschaftserkennung


Dieses Notebook dient als exploratives Hilfsmittel und dokumentiert die für die Hausarbeit des Moduls ***Computerlinguistische Programmierung*** entwickelten Skripte. Die decken die Datenaufbereitung, Modellierung und Auswertung ab. Die Skripte können entweder vom Notebook oder direkt von den einzelnen .py-Dateien ausgeführt werden.  

Alle Skripte, sowie eine Version der bearbeiteten Daten stehen in GitHub zur Verfügung:
https://github.com/RaikohGrass/partei-autorschaftserkennung

In [109]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Datenaufarbeitung

Die Rohdaten wurden vom [Open Discourse Projekt](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910%2FDVN%2FFIKIBO) genommen. Diese enthalten aufbereitete Plenarprotokollten des Deutschen Bundestages und eignen sich für die Anwengund und Untersuchung von NLP-Algorithmen.  

Für die Hausarbeit wurden folgende Datensätze heruntergeladen:
- ***speeches.feather***: diese Datei enthält die im Bundestag gehaltenen Reden seit 1949
- ***politicians.feather***: diese Datei enthält Informationen über die Mitglieder des Bundestages seit 1949
- ***factions.feather***: diese Datei enhält Information über die Parteien des Bundestages seit 1949

Da es nicht gewährleistet werden kann, dass die Ids der Dateien konstant bleiben, emptfehlen wir, die Dateien manuell herunterzuladen und im Subordner **"data/raw/....."** abzulegen.

Folgende Konstante werden als Konfiguration verwendet:

In [56]:
ELECTORAL_TERM = 19 # Wahlperiode
PARTIES = ["CDU/CSU", "SPD", "Grüne", "DIE LINKE.", "AfD"] # Zielparteien
SPEECH_MIN_WORD_COUNT = 100 # Mindestlänge der Reden
RANDOM_SEED = 11 # Replizierbarkeitreplizierbarkeit
TRAINING_SET_PERCENTAGE = 0.7 # Wieviel % der Daten werden fürs Training verwendet

### Filtern der Daten

Was wir erstmal machen wollen, ist die Daten zu filtern, so dass wir nur die gezielten Parteien und die Wahlperiode von Interesse berücksichtigen. Die Reden müssen auch eine Mindestlänge von 100 Wörtern haben.

In [57]:
from data_processing.filter_data import *

Erstmal werden die Rohdaten ausgelesen

In [58]:
df_speeches = pd.read_feather("data/raw/speeches.feather")
df_politicians = pd.read_feather("data/raw/politicians.feather")
df_factions = pd.read_feather("data/raw/factions.feather")

Erstmal suche ich die IDs der Parteien, die ich in die Analyse einbeziehen will.

In [59]:
factions_ids = df_factions[df_factions["abbreviation"].isin(PARTIES)][["id", "abbreviation"]].set_index("id").to_dict()["abbreviation"]

Danach filtere ich die Reden nach den IDs der Parteien und der Wahlperiode

In [60]:
df_speeches_filtered = filter_speeches(df_speeches, factions_ids, ELECTORAL_TERM, SPEECH_MIN_WORD_COUNT)
politicians_ids = df_speeches_filtered["politician_id"].unique().tolist()
df_politicians_filtered = df_politicians[df_politicians["id"].isin(politicians_ids)]

Jetzt teile ich das DataFrame in 3 Sätze (Trainingsatz, Testsatz und Validierungssatz)

In [61]:
df_train, df_val, df_test = split_data(df_speeches_filtered, TRAINING_SET_PERCENTAGE, r_seed=RANDOM_SEED)

Folgende Aufteilung nach Fraktion ist im Datensatz enthalten:

In [62]:
for name, dataset in [
    ("Training", df_train),
    ("Validierung", df_val),
    ("Test", df_test),
]:
    print(f"\n{name}:")
    print(dataset["faction_id"].value_counts())


Training:
faction_id
CDU/CSU       4483
SPD           3077
AfD           2646
Grüne         2249
DIE LINKE.    1778
Name: count, dtype: int64

Validierung:
faction_id
CDU/CSU       1020
SPD            695
DIE LINKE.     488
Grüne          465
AfD            378
Name: count, dtype: int64

Test:
faction_id
CDU/CSU       934
SPD           599
DIE LINKE.    542
AfD           434
Grüne         320
Name: count, dtype: int64


Da ich nur 1500 Rede pro Fraktion haben wollen, nehme ich eine Stichprobe vom Datensatz:

In [63]:
df_train_balanced = (
    df_train
    .groupby("faction_id", group_keys=False)
    .sample(n=1050, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

df_val_balanced = (
    df_val
    .groupby("faction_id", group_keys=False)
    .sample(n=225, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

df_test_balanced = (
    df_test
    .groupby("faction_id", group_keys=False)
    .sample(n=225, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

Jetzt muss es bestätigt werden, dass die Redner in den drei Datensätzen disjunkt sind.

In [64]:
proof_speaker_disjointness(df_train_balanced, df_val_balanced, df_test_balanced)

Finally we can save the filtered files

In [65]:
output_path = Path("data/filtered/train.feather")

output_path.parent.mkdir(parents=True, exist_ok=True)

df_train_balanced.to_feather("data/filtered/train.feather")
df_val_balanced.to_feather("data/filtered/val.feather")
df_test_balanced.to_feather("data/filtered/test.feather")

### Maskierung

Der nächste Schritt ist die Parteinamen in den Reden zu maskieren. Dafür eignet sich ein regex-Ausdruck, der alle die möglichen Namen erkennt:

In [66]:
from data_processing.process_data import mask_party_names

import re

PARTY_PATTERN = re.compile(
    r"\b(?:"
    r"CDU/CSU|CDU|CSU|"
    r"Unionsfraktion|Union|"
    r"SPD|"
    r"Sozialdemokrat(?:en|innen)?|"
    r"Sozialdemokratinnen und Sozialdemokraten|"
    r"sozialdemokratisch(?:e|en|er|es|em)?|"
    r"AfD|Alternative für Deutschland|"
    r"Bündnis 90/Die Grünen|Bündnis 90|"
    r"GRÜNE|Grüne|Grünen|Grüner|"
    r"Grünenfraktion|"
    r"DIE LINKE|Die Linke|Linke|Linken|Linker|Linksfraktion"
    r"Linksfraktion|"
    r"FDP|Freie Demokraten"
    r")\b"
)

für jeden Datensatz ersetzen wir die Parteinamen mit dem Token ***[PARTY]***

In [67]:
for dataset in [df_train_balanced, df_val_balanced, df_test_balanced]:
    dataset["masked_speech"] = (dataset["speech_content"].str.replace("\xa0", " ", regex=False).apply(mask_party_names, party_pattern=PARTY_PATTERN))

Wir speichern die aufbereiteten Datensätzen:

In [68]:
df_train_balanced.to_parquet("data/processed/train.parquet")
df_val_balanced.to_parquet("data/processed/val.parquet")
df_test_balanced.to_parquet("data/processed/test.parquet")

## Modelle trainieren und validieren

Ingesamt werden wir 4 Modelle trainieren:
- Baseline: einfach die häufigste Partei im Trainingsatz auswählen
- N-Gramme: mit Uni- und Bigrammen trainierte Logistische Regression
- Embeddings: Logistische Regression trainiert anhand Embeddings, die die Reden darstellen.
- LLM: ein BERT-Modell wird für die Aufgabe weitertrainiert

In [69]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

Erstens müssen wir die aufbereiteten Datensätze laden:

In [70]:
df_train = pd.read_parquet("data/processed/train.parquet")
df_validation = pd.read_parquet("data/processed/val.parquet")
df_test = pd.read_parquet("data/processed/test.parquet")

X_train = df_train["masked_speech"]
y_train = df_train["faction_id"]

X_validation = df_validation["masked_speech"]
y_validation = df_validation["faction_id"]

X_test = df_test["masked_speech"]
y_test = df_test["faction_id"]  

### Baseline

Da wir eine uniforme Verteilung der Reden gemacht haben, sollen das Baseline eine Genauigkeit von 20% (5 Parteien) haben. 

In [78]:
from models.baseline import baseline_model
m_baseline = baseline_model(X_train, y_train)

Wir machen die Vorhersage für den Validierungssatz mit dem Baseline-Modell

In [79]:
dummy_predictions = m_baseline.predict(X_validation)
print("dummy Validation accuracy:",accuracy_score(y_validation, dummy_predictions))

dummy Validation accuracy: 0.2


### N-Gramme

Für die N-Gramme haben wir einen TF-IDF-Vectorizer + eine logistische Regression ausgewählt.  

In [80]:
from models.ngram import ngram_model
m_ngram = ngram_model(X_train, y_train)
ngram_predictions = m_ngram.predict(X_validation)
print("ngram Validation accuracy:",accuracy_score(y_validation, ngram_predictions))

ngram Validation accuracy: 0.5875555555555556


### Embeddings

Der erste Schritt fürs Trainieren einer logistischen Regression mit Embeddings ist es, die Embeddings der Reden zu generieren. Dafür nutzen wir ein vortrainiertes Modell.  

Für die Hausarbeit wurde folgendes Modell ausgewählt:
***"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"***

Das Modell kann 150-Tokens-lang Sätze in einen Vektor mit 384 Dimensionen verwandeln und unterstützt [Deutsch](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2).

In [92]:
from data_processing.generate_embeddings import encode_speeches, embedding_model

Da das ausgewählte Modell nur Text mit einer Länge von 150 Tokens bearbeiten kann, müssen wir unsere Reden in Chunks aufteilen. Für jedes Chunk einer Rede wird ein Embedding berechnet und schließlich wird der Mittelwert aller dieser Embeddings als Embedding der Rede verwendet.

In [102]:
path_to_embeddings = Path("./data/embeddings/train_embeddings.npy")
# The embeddings will only be created if there are not found. The creation usually takes around 3 minutes
if path_to_embeddings.is_file():
    train_embeddings = np.load("data/embeddings/train_embeddings.npy")
    val_embeddings = np.load("data/embeddings/val_embeddings.npy")
    test_embeddings = np.load("data/embeddings/test_embeddings.npy")
else:
    train_embeddings = encode_speeches(X_train.tolist(), embedding_model)
    val_embeddings = encode_speeches(X_validation.tolist(), embedding_model)
    test_embeddings = encode_speeches(X_train.tolist(), embedding_model)

    np.save("data/embeddings/train_embeddings.npy", train_embeddings)
    np.save("data/embeddings/val_embeddings.npy", val_embeddings)
    np.save("data/embeddings/test_embeddings.npy", test_embeddings)

Mit den berechneten Embeddings können wir die logistische Regression trainieren

In [108]:
from models.embeddings import embedding_classifier
embedding_m = embedding_classifier(train_embeddings, y_train, regularization=100.0)
embedding_predictions = embedding_m.predict(val_embeddings)
print("Training accuracy:",accuracy_score(y_validation, embedding_predictions))

Training accuracy: 0.4008888888888889


### LLM

Das Ziel hier ist es, ein bereits trainiertes BERT-Modell für die Aufgabe weiterzutrainieren. Dafür wurde eine NVIDIA GPU 3080 verwendet. Ohne GPU wird es nicht empfohlen, die Trainingszellen dieses Notebooks auszuführen

#### Training

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from fine_tuning.fine_tuning import dataframe_to_chunks


classes = sorted(df_train["faction_id"].unique())
label2id = {label: i for i, label in enumerate(classes)}
id2label = {i: label for label, i in label2id.items()}

#### Load trained model

Hier wird das auf GitHub verfügbare Modell aufgeladen:

In [ ]:
FINAL_MODEL_PATH = "models/gbert-faction-classifier"
llm_tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_PATH)
llm_m = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_PATH)

In [ ]:
df_train_balanced = (
    df_train
    .groupby("faction_id", group_keys=False)
    .sample(n=1050, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

df_val_balanced = (
    df_val
    .groupby("faction_id", group_keys=False)
    .sample(n=225, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

df_test_balanced = (
    df_test
    .groupby("faction_id", group_keys=False)
    .sample(n=225, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

In [ ]:
df_train["masked_speech"].iloc[0]

'\n\nSehr geehrte Frau Präsidentin! Verehrte Kollegen! Liebe Negativzinssparer! Frau Lambrecht, vielleicht sollte ich angesichts der Märchenstunde, die Sie hier gehalten haben, lieber „Frau Grimm“ zu Ihnen sagen.\n\n({0})\n\nAls ich zum ersten Mal von den Fusionsplänen und dem Begleitschutz des Finanzministeriums im Zusammenhang mit diesen Fusionsplänen hörte, dachte ich sofort: Na, klasse! Diese linken Träumer von der [PARTY]! In deren Logik spannt man also neben einen lahmen Gaul einen weiteren lahmen Gaul vor die Postkutsche und denkt, man hätte einen Rennwagen. Ähnliches dachte man vielleicht auch bei Nahles und Scholz. Die Ergebnisse der Umfragen kennen Sie.\n\nNun aber versucht dieser Herr Scholz, eine deutsche Megabank entstehen zu lassen. Was sie tun soll, welches Ziel sie hätte – dazu bekam ich keine Antwort im Finanzausschuss, als ich fragte: Soll sie vielleicht die deutsche Finanzsouveränität und den deutschen Finanzplatz sichern? Soll sie eine feste Hand für deutsche Untern

In [ ]:
df_train["masked_speech"].iloc[0].split()

['Sehr',
 'geehrte',
 'Frau',
 'Präsidentin!',
 'Verehrte',
 'Kollegen!',
 'Liebe',
 'Negativzinssparer!',
 'Frau',
 'Lambrecht,',
 'vielleicht',
 'sollte',
 'ich',
 'angesichts',
 'der',
 'Märchenstunde,',
 'die',
 'Sie',
 'hier',
 'gehalten',
 'haben,',
 'lieber',
 '„Frau',
 'Grimm“',
 'zu',
 'Ihnen',
 'sagen.',
 '({0})',
 'Als',
 'ich',
 'zum',
 'ersten',
 'Mal',
 'von',
 'den',
 'Fusionsplänen',
 'und',
 'dem',
 'Begleitschutz',
 'des',
 'Finanzministeriums',
 'im',
 'Zusammenhang',
 'mit',
 'diesen',
 'Fusionsplänen',
 'hörte,',
 'dachte',
 'ich',
 'sofort:',
 'Na,',
 'klasse!',
 'Diese',
 'linken',
 'Träumer',
 'von',
 'der',
 '[PARTY]!',
 'In',
 'deren',
 'Logik',
 'spannt',
 'man',
 'also',
 'neben',
 'einen',
 'lahmen',
 'Gaul',
 'einen',
 'weiteren',
 'lahmen',
 'Gaul',
 'vor',
 'die',
 'Postkutsche',
 'und',
 'denkt,',
 'man',
 'hätte',
 'einen',
 'Rennwagen.',
 'Ähnliches',
 'dachte',
 'man',
 'vielleicht',
 'auch',
 'bei',
 'Nahles',
 'und',
 'Scholz.',
 'Die',
 'Ergebniss

## Benchmarking

## Ergebnisanalyse

In [84]:
vectorizer = m_ngram.named_steps["tfidf"]
classifier = m_ngram.named_steps["classifier"]

In [85]:
feature_names = np.array(vectorizer.get_feature_names_out())